# SageMaker Setup — Entity Resolution Pipeline

Run cells top to bottom on a SageMaker notebook instance.

**Prereqs:**
- Upload the `erpipeline/` folder to `/home/ec2-user/SageMaker/erpipeline`
  (or `git clone` your repo there).
- Upload the dataset so this path exists:
  `/home/ec2-user/SageMaker/erpipeline/data/raw/train/train_source1.tsv`
  (copy from `student_resource/dataset/`, or `aws s3 cp` it down).
- Suggested instance: `ml.t3.medium` for smoke tests,
  `ml.m5.4xlarge`+ for larger runs, `ml.g4dn.xlarge` for embedding speed.

In [ ]:
# 0. Work from the project root
%cd /home/ec2-user/SageMaker/erpipeline
%pwd

In [ ]:
# 1. Install dependencies into the notebook kernel
# (pypostal-multiarch installs as a wheel; if libpostal data is missing,
#  the pipeline automatically falls back to regex address parsing.)
%pip install -r requirements.txt

In [ ]:
# 2. Verify all libraries import correctly
%run test_libraries.py

In [ ]:
# 3. Point config paths at SageMaker locations
import yaml
from pathlib import Path

ROOT = Path('/home/ec2-user/SageMaker/erpipeline')
cfg_path = ROOT / 'config.yaml'
cfg = yaml.safe_load(cfg_path.read_text())

cfg['paths']['raw_dir'] = str(ROOT / 'data' / 'raw')
cfg['paths']['processed_dir'] = str(ROOT / 'data' / 'processed')
cfg['paths']['candidates_dir'] = str(ROOT / 'data' / 'candidates')
cfg['paths']['models_dir'] = str(ROOT / 'models')
cfg['paths']['output_dir'] = str(ROOT / 'output')
cfg['validation']['test_dir'] = str(ROOT / 'data' / 'raw' / 'test')

cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print('config.yaml updated:')
print(yaml.safe_dump({'paths': cfg['paths']}, sort_keys=False))

# sanity-check dataset presence
for split in ['train', 'test']:
    for src in ['source1', 'source2', 'source3']:
        p = ROOT / 'data' / 'raw' / split / f'{split}_{src}.tsv'
        print(('OK  ' if p.exists() else 'MISS'), p)

In [ ]:
# 4. Smoke test: preprocess 3 records through the real pipeline
from src.preprocessing.pipeline import PreprocessingPipeline
from src.config import load_config

config = load_config('config.yaml')
pipe = PreprocessingPipeline(config)
rows = [
    {'entity_id': 'S1-1', 'business_name': 'Waters & Millar Aimei P.C.',
     'business_address': '6267 Passing Sky Drive, Security-widefield, CO',
     'country': 'US'},
    {'entity_id': 'S2-1', 'business_name': 'Aimei Waters & Millar P.C.',
     'business_address': '6267B PASSING SKY DR, SECURITY-WIDEFIELD, CO',
     'country': 'US'},
    {'entity_id': 'S2-2', 'business_name': 'एसएस फूड प्राइवेट लिमिटेड',
     'business_address': 'AF-0684, NANDGRAM, GHAZIABAD, उत्तर प्रदेश',
     'country': 'India'},
]
for r in rows:
    out = pipe.process_record(r)
    print(out['entity_id'], '->', repr(out['name_clean']),
          '| PINs:', out['postal_codes'],
          '| metaphone_sorted:', repr(out['metaphone_sorted']))

In [ ]:
# 5. Small end-to-end run (50 S1 entities, ~5-10 min on CPU)
# Full output is also piped to a timestamped log file.
%%bash
cd /home/ec2-user/SageMaker/erpipeline
LOG=~/test_output_sagemaker_$(date +%Y%m%d_%H%M%S).log
python test_full_pipeline.py 2>&1 | tee "$LOG"
echo "LOGGED TO: $LOG"

## Scaling up

- **Bigger samples**: edit `SAMPLE_SIZE` / `TEST_SIZE` at the top of
  `test_full_pipeline.py` (validated at 2000 / 1800-200).
- **Full data**: run `python -m src.main --stage all` — use a
  `ml.m5.4xlarge`+ instance with large EBS; Polars streams, but the
  blocking indexes for 10M records need RAM.
- **Embeddings**: `ml.g4dn.xlarge` cuts bi-encoder time sharply;
  switch `faiss-cpu` to `faiss-gpu` only on GPU instances.
- **Long jobs**: move runs over ~1h to a SageMaker Training or
  Processing Job so notebook timeouts can't kill them.
- **Optional libpostal**: notebook instances allow `sudo`, so
  `sudo apt-get install build-essential autoconf automake libtool`
  + building libpostal from source enables full address parsing;
  otherwise the regex fallback is used automatically.